# Let decision models guide your voice assistant

A cleaner reports observations during a rental apartment inspection. **Decision-1** assesses four requirements and selects the next conversational step in one request. **GPT-Live** listens and speaks. Python collects the conversation and forwards the model answers unchanged.

The five questions share the conversation. We do not explicitly pass the returned statuses into the next-step question. Every request offers the complete choice set.

Run this notebook to explore a seven-update scenario with scripted synthetic cleaner speech and actual GPT-Live replies. The saved walkthrough makes no API calls and does not activate the microphone. Azure credentials are needed only for the optional live calls.

## The four requirements

| Check | Required condition |
|---|---|
| Entrance | The current guest access code successfully opens the entrance. |
| Bedroom | The bed is made with clean sheets and clean pillowcases. |
| Bathroom sink | The sink drains normally and stays dry underneath while water runs. |
| Kitchen | Counters are clean and the refrigerator has no previous guests' food. |

All requirements share three status definitions: `ready`, `issue`, and `needs_evidence`. A determined assessment can contain a problem; `finish` does not mean everything passed.

## Set up the Azure connections

Use Python 3.11+ and install `requirements.txt`. To make live calls, copy `.env.example` to `.env` and fill in your shared Azure Foundry endpoint and key, plus the deployment names. Without credentials, the notebook still displays the saved scenario.

The HTTP client uses the operating system certificate store. No connection is made until a live function is called.

In [1]:
import asyncio
import base64
import json
import os
import ssl
import time
from pathlib import Path

import httpx
import numpy as np
from dotenv import load_dotenv
from IPython.display import Audio, Markdown, display
from websockets.asyncio.client import connect

load_dotenv(override=True)
AZURE_ENDPOINT = os.environ.get('AZURE_ENDPOINT', 'https://YOUR-RESOURCE.services.ai.azure.com').rstrip('/')
API_KEY = os.environ.get('AZURE_API_KEY', '')
DECISION_MODEL = os.environ.get('AZURE_DECISION_DEPLOYMENT', 'Decision-1')
LIVE_MODEL = os.environ.get('AZURE_LIVE_DEPLOYMENT', 'gpt-live-1')
LIVE_URL = AZURE_ENDPOINT.replace('https://', 'wss://') + '/openai/v1/live/sessions'
http = httpx.AsyncClient(headers={'api-key': API_KEY}, verify=ssl.create_default_context(), timeout=30)


## Define the decisions

`CHECKS` contains the task conditions. `STATUS_CRITERIA` is one shared set of meanings: sufficient current evidence, a current observed failure, or insufficient evidence. `EVIDENCE_POLICY` handles attribution, chronology, corrections, and incomplete verification in general terms.

Each named entry in `QUESTIONS` is a native choice question with `type`, `instructions`, and `criteria`. For example, `QUESTIONS['bathroom']` evaluates the sink. All four status questions share the same conversation in one request.

In [2]:
CHECKS = {
    "entrance": "The current guest access code has successfully opened the entrance door.",
    "bedroom": "The bed is made with clean sheets and clean pillowcases.",
    "bathroom": "The sink drains normally and its underside stays dry while water is running.",
    "kitchen": "The counters are clean and the refrigerator contains no food left by previous guests."
}

STATUS_CRITERIA = {
    "needs_evidence": "Current reported evidence is insufficient to establish every part of the requirement, and does not establish a current failure.",
    "ready": "Current reported evidence establishes every part of the requirement.",
    "issue": "Current reported evidence establishes an unresolved failure of at least one part of the requirement."
}

EVIDENCE_POLICY = (
    "Assess the current condition from the accumulated conversation. Use the cleaner's reports "
    'of observations and completed actions as evidence. Assistant statements supply '
    'conversational context, not independent observations. Interpret short replies in the '
    'context of the question they answer. Apply corrections to the facts they concern and '
    'preserve unrelated valid observations. Respect the chronology of events, not merely the '
    'order of sentences. Evidence must still apply to the current condition; a material '
    'intervening change can make an earlier verification insufficient. A proposed action, '
    'intended action, or unverified general assurance does not establish its required result. '
    'Every condition joined by AND is necessary for ready. Distinguish unknown evidence from an'
    ' observed failure. '
)

QUESTIONS = {
    name: {'type': 'choice', 'instructions': EVIDENCE_POLICY + 'Requirement: ' + requirement,
           'criteria': STATUS_CRITERIA}
    for name, requirement in CHECKS.items()
}


## Build the decision function

Add a fifth question for the next conversational step. It receives all four requirements and the same general evidence policy. Its six choices are always available: verify one of the four requirements, defer because verification is blocked, or summarize the assessment.

`assess_inspection()` sends all five questions in one request and returns the selected labels. Python neither interprets the observations nor changes the model's answers. `finish` means the assessment is determined, including any known issues; it does not mean the apartment is ready.

In [3]:
ACTION_POLICY = 'Choose the next useful conversational step from the conversation and all four requirements. Use the same evidence policy to determine which requirements still lack sufficient evidence. Request verification of an unresolved requirement that the cleaner can advance now, considering current access, timing, available people and equipment. There is no fixed room order; if several checks are feasible, any useful feasible check is acceptable. An assistant suggestion does not establish feasibility. Choose defer when evidence is still missing but none of the remaining requirements can be advanced now. Choose finish when every requirement has a determined outcome, including known problems; finish means summarize the assessment, not that the apartment is ready.'

NEXT_ACTIONS = {
    f'ask_{name}': 'Collect the missing evidence for: ' + requirement
    for name, requirement in CHECKS.items()
}
NEXT_ACTIONS['defer'] = 'Explain that remaining evidence cannot be collected now and identify the blocker to resuming.'
NEXT_ACTIONS['finish'] = 'Summarize the determined outcomes, including any known problems. Readiness requires every requirement to be met.'

QUESTIONS['next_action'] = {
    'type': 'choice',
    'instructions': EVIDENCE_POLICY + ACTION_POLICY + '\nRequirements: ' + json.dumps(CHECKS),
    'criteria': NEXT_ACTIONS,
}


async def assess_inspection(conversation):
    response = await http.post(
        f'{AZURE_ENDPOINT}/providers/microsoft/v1/systemone',
        json={'model': DECISION_MODEL, 'state': {'conversation': conversation},
              'questions': QUESTIONS},
    )
    response.raise_for_status()
    return {name: answer['choice'] for name, answer in response.json()['answers'].items()}


def show_assessment(result):
    display(Markdown('| Entrance | Bedroom | Bathroom sink | Kitchen |\n'
                     '|---|---|---|---|\n| ' + ' | '.join(result[name] for name in CHECKS) + ' |'))
    print('Next conversational step:', result['next_action'])


## Set up the voice conversation

GPT-Live supplies the input and output transcripts as well as generated speech. These instructions make it delegate new observations and corrections rather than decide inspection outcomes itself. `append_transcript()` joins adjacent fragments by the same speaker so context survives across short replies.

In [4]:
LIVE_INSTRUCTIONS = 'You are the spoken interface for a rental apartment inspection. Speak English only.\nDecision-1 assesses the requirements and selects the next conversational step. You express its answers naturally.\nKeep replies brief and ask at most one question. While backend work is pending, only acknowledge briefly.\nGenerate <|SpawnThinking|> when the cleaner finishes an observation, answers a question, or corrects facts or practical constraints.\nDo not delegate for greetings, while the cleaner is still listing information, or merely because a backend result arrives.\nWait for backend answers before asking an inspection question or reporting inspection outcomes.\nFollow next_action; use the conversation to phrase the question about the missing evidence without inventing observations.\nStatus meanings: ready means the requirement is met; issue means a confirmed problem; needs_evidence means insufficient evidence to determine the requirement.\nFor defer, explain the blocker from the conversation and wait. For finish, summarize the returned outcomes, including any problems. Only describe the apartment as ready when all four requirements are ready.\n' + '\nRequirements: ' + json.dumps(CHECKS) + '\nNext-step meanings: ' + json.dumps(NEXT_ACTIONS)

def append_transcript(conversation, speaker, text):
    if conversation and conversation[-1]['speaker'] == speaker:
        conversation[-1]['text'] += text
    else:
        conversation.append({'speaker': speaker, 'text': text})


## Connect decisions to spoken replies

`run_voice()` streams audio and receives transcript, delegation, and speech events. When GPT-Live requests an assessment, Python sends the accumulated conversation to Decision-1 once and forwards its selected labels unchanged. GPT-Live already has the requirements and label meanings in its instructions.

A short audio/transcript quiet interval lets the cleaner finish speaking before assessment. Audio returned by GPT-Live goes to `on_audio` for playback or recording.

In [5]:
async def run_voice(audio_source, on_audio=None):
    conversation, assessments = [], []
    last_input = last_speech = 0.0
    pending = None
    async with connect(LIVE_URL, additional_headers={'api-key': API_KEY}) as ws:
        await ws.send(json.dumps({'type': 'session.start', 'session': {
            'model': LIVE_MODEL, 'instructions': LIVE_INSTRUCTIONS,
            'audio': {'output': {'voice': 'marin'}}, 'delegation': {'type': 'client'},
        }}))
        started = json.loads(await ws.recv())
        assert started['type'] == 'session.started', started

        async def decide(delegation_id):
            while time.perf_counter() - max(last_input, last_speech) < 0.9:
                await asyncio.sleep(0.1)
            assessment = await assess_inspection(conversation)
            assessments.append(assessment)
            await ws.send(json.dumps({'type': 'session.commentary.append',
                'delegation_id': delegation_id, 'content': json.dumps(assessment)}))
            show_assessment(assessment)

        async def send_audio():
            nonlocal last_speech
            async for pcm in audio_source:
                if np.abs(np.frombuffer(pcm, dtype=np.int16).astype(np.int32)).max() > 500:
                    last_speech = time.perf_counter()
                await ws.send(json.dumps({'type': 'session.input_audio.append',
                    'audio': base64.b64encode(pcm).decode()}))
            if pending:
                await pending
            await ws.send(json.dumps({'type': 'session.close'}))

        sender = asyncio.create_task(send_audio())
        async for raw in ws:
            event = json.loads(raw)
            kind = event['type']
            if kind == 'session.input_transcript.delta':
                append_transcript(conversation, 'cleaner', event['delta'])
                last_input = time.perf_counter()
            elif kind == 'session.output_transcript.delta':
                append_transcript(conversation, 'assistant', event['delta'])
            elif kind == 'session.delegation.created':
                if pending and not pending.done():
                    pending.cancel()
                pending = asyncio.create_task(decide(event['delegation']['id']))
            elif kind == 'session.output_audio.delta' and on_audio:
                on_audio(base64.b64decode(event['delta']))
            elif kind == 'error':
                raise RuntimeError(event['error'])
            elif kind == 'session.closed':
                break
        await sender
    return {'conversation': conversation, 'assessments': assessments}


## Run the recorded scenario

The cleaner's input is scripted synthetic speech. The assistant transcripts, decisions, and audio below are the actual outputs of the seven-update session. The scenario includes changing access, a leak, a correction to the bedding report, and a successful retest.

Decision-1 received the actual conversation transcripts. Each update used one decision request. Audio players replay the saved assistant speech; the cleaner recordings are also included under `audio/`.

In [6]:
recorded = json.loads(Path('results/voice_run.json').read_text(encoding='utf-8'))
for index, turn in enumerate(recorded['turns'], 1):
    display(Markdown(f"**Update {index}.** {turn['cleaner_text']}"))
    show_assessment(turn['assessment'])
    display(Markdown('**Assistant:** ' + turn['assistant_text']))
    display(Audio(url=turn['output_audio'], embed=False))

# Optional: make a fresh Decision-1 call using the first recorded input.
# fresh = await assess_inspection([
#     {'speaker': 'cleaner', 'text': recorded['turns'][0]['cleaner_text']}
# ])
# show_assessment(fresh)


**Update 1.** The current guest code opened the entrance I made the bed with clean sheets and clean pillowcases I have not inspected the sink or kitchen yet The kitchen is closed while the painter works there. But I can inspect the bathroom now

| Entrance | Bedroom | Bathroom sink | Kitchen |
|---|---|---|---|
| ready | ready | needs_evidence | needs_evidence |

Next conversational step: ask_bathroom


**Assistant:** Okay. Got it. Okay. Thanks for letting me know. When you run the bathroom sink, does it drain normally, and stay dry underneath?

**Update 2.** Hold on. The plumber has just shut off the bathroom water for the next hour The painter has has finished and reopened the kitchen. I still have not checked either room. But I can go into the kitchen now

...

**Assistant:** Okay. The place still isn’t ready yet. The bathroom sink is leaking. The other areas look good.

**Update 7.** Maintenance has now repaired the joint. After they finished, I ran the tap for a full minute. Water drained normally, and I watched underneath the whole time. It stayed dry

| Entrance | Bedroom | Bathroom sink | Kitchen |
|---|---|---|---|
| ready | ready | ready | ready |

Next conversational step: finish


**Assistant:** Great. Everything’s ready now.

## Try your microphone

Uncomment the last call for a two-minute microphone session on the local machine. Use headphones. The experiment exercised the synthetic audio path; this microphone adapter has not been tested with human speech. Starting it makes real Azure model calls.

In [7]:
async def microphone(seconds):
    import sounddevice as sd
    queue = asyncio.Queue()
    loop = asyncio.get_running_loop()

    def capture(indata, frames, timing, status):
        loop.call_soon_threadsafe(queue.put_nowait, bytes(indata))

    with sd.RawInputStream(samplerate=24000, channels=1, dtype='int16',
                           blocksize=960, callback=capture):
        end = loop.time() + seconds
        while loop.time() < end:
            yield await queue.get()


async def inspect_property(seconds=120):
    import sounddevice as sd
    audio_queue = asyncio.Queue()
    with sd.RawOutputStream(samplerate=24000, channels=1, dtype='int16') as speaker:
        async def play():
            while (pcm := await audio_queue.get()) is not None:
                await asyncio.to_thread(speaker.write, pcm)

        playback = asyncio.create_task(play())
        result = await run_voice(microphone(seconds), on_audio=audio_queue.put_nowait)
        await audio_queue.put(None)
        await playback
    return result


# live_result = await inspect_property(seconds=120)


## Apply the pattern

Define requirements and allowed choices, give the decision model the conversation, and return its selected labels to the voice model. To adapt the example, start with `CHECKS`, the evidence policy, and the next-step choices.

The complete recorded exchange is in `results/voice_run.json`. The architecture diagram and status timeline in `results/` show how the components and decisions fit together.